# ⚡ Async Programming for LLM Applications

**Speed up API calls with concurrent execution**

---

## 📋 Overview

**What you'll learn:**
- async/await fundamentals
- Concurrent LLM API calls
- Batch processing documents
- Error handling in async code
- Performance comparison

**Time estimate:** ⏱️ 50 minutes | **Difficulty:** 🟡 Intermediate

---

In [ ]:
import asyncio
import time
from typing import List
import aiohttp

print("✅ Setup complete")

## 🤔 Why Async for LLM Apps?

**Problem: API calls are slow**
```python
# Sequential (SLOW)
result1 = call_llm(prompt1)  # 2 seconds
result2 = call_llm(prompt2)  # 2 seconds
result3 = call_llm(prompt3)  # 2 seconds
# Total: 6 seconds 😴
```

**Solution: Async (FAST)**
```python
# Concurrent (FAST)
results = await asyncio.gather(
    call_llm_async(prompt1),
    call_llm_async(prompt2),
    call_llm_async(prompt3)
)
# Total: 2 seconds ⚡ (3x faster!)
```

**Use cases:**
- 🔄 Processing multiple documents
- 🌐 Multiple API calls
- 📊 Batch embeddings
- 🔍 Parallel RAG retrieval

## 🎯 Async Basics

In [ ]:
# Simple async function
async def fetch_data(n: int):
    """Simulate API call."""
    print(f"  Starting fetch {n}...")
    await asyncio.sleep(1)  # Simulate network delay
    print(f"  Finished fetch {n}")
    return f"Data {n}"

# Run single async function
result = await fetch_data(1)
print(f"Result: {result}")

In [ ]:
# Sequential vs Concurrent comparison
print("📊 Sequential (one at a time):")
start = time.time()
results = []
for i in range(3):
    result = await fetch_data(i)
    results.append(result)
sequential_time = time.time() - start
print(f"Time: {sequential_time:.2f}s\n")

print("⚡ Concurrent (all at once):")
start = time.time()
results = await asyncio.gather(
    fetch_data(0),
    fetch_data(1),
    fetch_data(2)
)
concurrent_time = time.time() - start
print(f"Time: {concurrent_time:.2f}s")

print(f"\n🚀 Speedup: {sequential_time/concurrent_time:.1f}x faster!")

## 🤖 Async LLM API Calls

In [ ]:
from openai import AsyncOpenAI
import os

# Initialize async client
client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))

async def call_llm_async(prompt: str, model: str = "gpt-3.5-turbo"):
    """Async LLM call."""
    try:
        response = await client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=50
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Test single call
result = await call_llm_async("Say 'Hello' in Spanish")
print(f"Result: {result}")

In [ ]:
# Batch processing prompts
prompts = [
    "Translate 'Hello' to Spanish",
    "Translate 'Hello' to French",
    "Translate 'Hello' to German",
    "Translate 'Hello' to Italian",
    "Translate 'Hello' to Japanese",
]

print("⚡ Processing 5 prompts concurrently...\n")
start = time.time()

# Run all concurrently
results = await asyncio.gather(*[call_llm_async(p) for p in prompts])

elapsed = time.time() - start

print("Results:")
for prompt, result in zip(prompts, results):
    print(f"  {prompt[:30]:30} → {result}")

print(f"\n⏱️ Total time: {elapsed:.2f}s")
print(f"📊 Average per call: {elapsed/len(prompts):.2f}s")

## 🔄 Batch Document Processing

In [ ]:
class AsyncDocumentProcessor:
    """Process documents concurrently."""
    
    def __init__(self, max_concurrent: int = 5):
        self.max_concurrent = max_concurrent
        self.client = AsyncOpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    
    async def process_document(self, doc: str, task: str) -> dict:
        """Process single document."""
        prompt = f"{task}\n\nDocument: {doc}"
        
        try:
            response = await self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=100
            )
            
            return {
                'document': doc[:50],
                'result': response.choices[0].message.content,
                'tokens': response.usage.total_tokens,
                'status': 'success'
            }
        except Exception as e:
            return {
                'document': doc[:50],
                'result': None,
                'error': str(e),
                'status': 'error'
            }
    
    async def process_batch(self, documents: List[str], task: str) -> List[dict]:
        """Process batch with concurrency limit."""
        semaphore = asyncio.Semaphore(self.max_concurrent)
        
        async def limited_process(doc):
            async with semaphore:
                return await self.process_document(doc, task)
        
        tasks = [limited_process(doc) for doc in documents]
        results = await asyncio.gather(*tasks)
        
        return results

# Test it
processor = AsyncDocumentProcessor(max_concurrent=3)

documents = [
    "The quick brown fox jumps over the lazy dog.",
    "Python is a great programming language.",
    "Machine learning is transforming industries.",
    "Cloud computing enables scalable applications.",
]

print("📄 Processing 4 documents (max 3 concurrent)...\n")
start = time.time()

results = await processor.process_batch(documents, "Summarize in 5 words")

elapsed = time.time() - start

print("Results:")
for r in results:
    status = "✅" if r['status'] == 'success' else "❌"
    print(f"  {status} {r['document'][:40]:40} → {r.get('result', r.get('error'))}")

print(f"\n⏱️ Total time: {elapsed:.2f}s")

## 🛡️ Error Handling in Async

In [ ]:
async def safe_api_call(prompt: str, retries: int = 3):
    """API call with retry logic."""
    for attempt in range(retries):
        try:
            response = await client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=50
            )
            return response.choices[0].message.content
        
        except Exception as e:
            if attempt < retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff
                print(f"  Attempt {attempt + 1} failed. Retrying in {wait_time}s...")
                await asyncio.sleep(wait_time)
            else:
                print(f"  All {retries} attempts failed")
                raise

# Test
try:
    result = await safe_api_call("Hello!")
    print(f"✅ Success: {result}")
except Exception as e:
    print(f"❌ Failed: {e}")

## ⚡ Advanced: Rate Limiting

In [ ]:
import asyncio
from collections import deque
from datetime import datetime, timedelta

class RateLimiter:
    """Rate limiter for API calls."""
    
    def __init__(self, max_calls: int, time_window: float):
        """
        Args:
            max_calls: Maximum calls allowed
            time_window: Time window in seconds
        """
        self.max_calls = max_calls
        self.time_window = time_window
        self.calls = deque()
    
    async def acquire(self):
        """Wait until we can make a call."""
        now = datetime.now()
        
        # Remove old calls outside window
        cutoff = now - timedelta(seconds=self.time_window)
        while self.calls and self.calls[0] < cutoff:
            self.calls.popleft()
        
        # If at limit, wait
        if len(self.calls) >= self.max_calls:
            sleep_time = (self.calls[0] - cutoff).total_seconds()
            await asyncio.sleep(sleep_time)
            return await self.acquire()
        
        # Record this call
        self.calls.append(now)

# Example: Max 3 calls per 2 seconds
limiter = RateLimiter(max_calls=3, time_window=2.0)

async def rate_limited_call(n: int):
    """Make rate-limited API call."""
    await limiter.acquire()
    print(f"  Call {n} at {datetime.now().strftime('%H:%M:%S')}")
    return f"Result {n}"

print("🔄 Making 6 calls (rate limit: 3 per 2s)...\n")
start = time.time()

results = await asyncio.gather(*[
    rate_limited_call(i) for i in range(6)
])

elapsed = time.time() - start
print(f"\n⏱️ Total time: {elapsed:.2f}s")
print("Notice: First 3 calls instant, then 2s wait, then 3 more calls")

## ✅ Summary

### Key Concepts:
- `async def` - Define async function
- `await` - Wait for async operation
- `asyncio.gather()` - Run multiple tasks concurrently
- `asyncio.Semaphore()` - Limit concurrency
- Exponential backoff for retries

### Performance Gains:
```
Sequential:  N × API_TIME
Concurrent:  API_TIME (if unlimited concurrency)
Speedup:     Up to N× faster!
```

### When to Use Async:
- ✅ **Multiple API calls**: 3-10x speedup
- ✅ **Batch processing**: Documents, embeddings
- ✅ **I/O bound tasks**: Network, disk operations
- ❌ **CPU bound tasks**: Use multiprocessing instead

### Best Practices:
1. **Limit concurrency**: Use `Semaphore` (5-10 concurrent)
2. **Handle errors**: Try/except + retries
3. **Rate limiting**: Respect API limits
4. **Exponential backoff**: 1s, 2s, 4s, 8s delays

### Production Tips:
- 🎯 **Semaphore limit**: 5-10 for most APIs
- ⚡ **Batch size**: 10-50 items optimal
- 🔄 **Retry logic**: 3 attempts with backoff
- 📊 **Monitor**: Track success/failure rates

### Next: `02_llm_basics/05_error_handling.ipynb`